<a href="https://colab.research.google.com/github/vixiborges/GustavoBorges_rm567477/blob/main/GustavoBorges_rm567477_pbl_fase4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌾 FarmTech Solutions — Previsão de Rendimento de Safra

**Projeto:** Machine Learning para Análise Agrícola  
**Dataset:** `crop_yield.csv`  
**Objetivo:** Explorar tendências de produtividade via clusterização e prever o rendimento de safra com 5 algoritmos de regressão supervisionada.

---

## 📋 Sumário

1. [Importação de Bibliotecas](#1)
2. [Carregamento e Análise Exploratória dos Dados (EDA)](#2)
3. [Pré-processamento](#3)
4. [Clusterização (Machine Learning Não Supervisionado)](#4)
5. [Modelos Preditivos de Regressão (Machine Learning Supervisionado)](#5)
6. [Comparação dos Modelos](#6)
7. [Conclusões](#7)

---
## 1. Importação de Bibliotecas <a id='1'></a>

In [ ]:
# ── Manipulação de dados ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualização ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Pré-processamento ─────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score

# ── Clusterização (Não Supervisionado) ────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ── Modelos de Regressão (Supervisionado) ─────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# ── Métricas de avaliação ─────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Configurações visuais ─────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14

import warnings
warnings.filterwarnings('ignore')  # Suprime avisos de versão das bibliotecas

print('✅ Bibliotecas importadas com sucesso!')

✅ Bibliotecas importadas com sucesso!


---
## 2. Carregamento e Análise Exploratória dos Dados (EDA) <a id='2'></a>

A **Análise Exploratória de Dados (EDA)** é o primeiro passo de qualquer projeto de Machine Learning. Aqui vamos entender a estrutura do dataset, identificar valores ausentes, distribuições e relações entre variáveis.

In [ ]:
# Carregamento do dataset
df = pd.read_csv('crop_yield.csv')

# Exibindo as primeiras linhas para entender a estrutura
print('🔍 Primeiras linhas do dataset:')
df.head(10)

🔍 Primeiras linhas do dataset:


,Crop,Precipitation (mm day-1),Specific Humidity at 2 Meters (g/kg),Relative Humidity at 2 Meters (%),Temperature at 2 Meters (C),Yield
0,"Cocoa, beans",2248.92,17.72,83.40,26.01,11560
1,"Cocoa, beans",1938.42,17.54,82.11,26.11,11253
2,"Cocoa, beans",2301.54,17.81,82.79,26.24,9456
3,"Cocoa, beans",2592.35,17.61,85.07,25.56,9321
4,"Cocoa, beans",2344.72,17.61,84.12,25.76,8800
5,"Cocoa, beans",2339.30,17.70,84.54,25.76,8850
6,"Cocoa, beans",2326.09,18.09,84.63,26.11,9003
7,"Cocoa, beans",2718.08,18.30,85.43,26.12,9880
8,"Cocoa, beans",2061.61,17.80,84.36,25.88,9201
9,"Cocoa, beans",1934.62,17.94,83.43,26.21,8300


In [ ]:
# Dimensões do dataset: linhas x colunas
print(f'📊 Dimensões do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas')

# Tipos de dados de cada coluna
print('\n📋 Tipos de dados:')
print(df.dtypes)

📊 Dimensões do dataset: 156 linhas x 6 colunas

📋 Tipos de dados:
Crop                                     object
Precipitation (mm day-1)                float64
Specific Humidity at 2 Meters (g/kg)    float64
Relative Humidity at 2 Meters (%)       float64
Temperature at 2 Meters (C)             float64
Yield                                     int64
dtype: object


In [ ]:
# Verificação de valores nulos — fundamental antes de qualquer modelagem
print('🔎 Valores nulos por coluna:')
print(df.isnull().sum())
print(f'\nTotal de valores nulos: {df.isnull().sum().sum()}')

🔎 Valores nulos por coluna:
Crop                                    0
Precipitation (mm day-1)                0
Specific Humidity at 2 Meters (g/kg)    0
Relative Humidity at 2 Meters (%)       0
Temperature at 2 Meters (C)             0
Yield                                   0
dtype: int64

Total de valores nulos: 0


In [ ]:
# Culturas únicas presentes no dataset
print('🌱 Culturas presentes no dataset:')
print(df['Crop'].value_counts())

🌱 Culturas presentes no dataset:
Crop
Cocoa, beans       39
Oil palm fruit     39
Rice, paddy        39
Rubber, natural    39
Name: count, dtype: int64


In [ ]:
# Estatísticas descritivas: média, desvio padrão, mín, máx etc.
print('📈 Estatísticas descritivas:')
df.describe().round(2)

### 2.1 Distribuição das variáveis numéricas

In [ ]:
# Histogramas para visualizar a distribuição de cada variável numérica
colunas_num = ['Precipitation (mm day-1)',
               'Specific Humidity at 2 Meters (g/kg)',
               'Relative Humidity at 2 Meters (%)',
               'Temperature at 2 Meters (C)',
               'Yield']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(colunas_num):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frequência')

axes[-1].set_visible(False)  # Remove o subplot extra
fig.suptitle('Distribuição das Variáveis Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Boxplots — Identificação de Outliers

In [ ]:
# Boxplots por cultura para identificar outliers no rendimento (Yield)
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Crop', y='Yield', palette='Set2')
plt.title('Distribuição do Rendimento (Yield) por Cultura', fontweight='bold')
plt.xlabel('Cultura')
plt.ylabel('Rendimento (Yield)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 2.3 Mapa de Correlação

In [ ]:
# Mapa de calor das correlações entre variáveis numéricas
# Correlação indica o quanto uma variável está relacionada linearmente com outra
plt.figure(figsize=(9, 6))
corr = df[colunas_num].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, vmin=-1, vmax=1)
plt.title('Mapa de Correlação das Variáveis', fontweight='bold')
plt.tight_layout()
plt.show()

### 2.4 Rendimento médio por cultura

In [ ]:
# Gráfico de barras: rendimento médio por tipo de cultura
media_por_cultura = df.groupby('Crop')['Yield'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
media_por_cultura.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Rendimento Médio por Cultura', fontweight='bold')
plt.xlabel('Cultura')
plt.ylabel('Rendimento Médio (Yield)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print(media_por_cultura)

---
## 3. Pré-processamento <a id='3'></a>

Antes de alimentar os dados nos modelos, precisamos:
- **Codificar** variáveis categóricas (nomes das culturas) em números
- **Normalizar** as variáveis numéricas, para que nenhuma variável domine as outras por ter escala maior
- **Separar** os dados em treino e teste

In [ ]:
# Cópia do dataframe original para não modificá-lo
df_proc = df.copy()

# Label Encoding: converte os nomes das culturas em números inteiros
# Ex: 'Cocoa, beans' → 0, 'Oil palm fruit' → 1, etc.
le = LabelEncoder()
df_proc['Crop_encoded'] = le.fit_transform(df_proc['Crop'])

print('📌 Mapeamento de culturas:')
for i, classe in enumerate(le.classes_):
    print(f'  {i} → {classe}')

In [ ]:
# Definição das features (X) e do alvo (y)
# X: variáveis de entrada que o modelo usará para aprender
# y: variável que queremos prever (rendimento)

features = ['Crop_encoded',
            'Precipitation (mm day-1)',
            'Specific Humidity at 2 Meters (g/kg)',
            'Relative Humidity at 2 Meters (%)',
            'Temperature at 2 Meters (C)']

X = df_proc[features]
y = df_proc['Yield']

print(f'✅ X shape: {X.shape}')   # (156 amostras, 5 features)
print(f'✅ y shape: {y.shape}')   # (156 alvos)

In [ ]:
# Normalização via StandardScaler: transforma os dados para média 0 e desvio 1
# Importante para algoritmos sensíveis à escala (ex: SVR)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Separação treino/teste: 80% para treinar, 20% para testar
# random_state garante reprodutibilidade dos resultados
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f'🔀 Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')

---
## 4. Clusterização — Machine Learning Não Supervisionado <a id='4'></a>

A clusterização agrupa os dados em **clusters (grupos)** sem que o modelo saiba previamente as categorias corretas. Usaremos o algoritmo **K-Means**, que busca minimizar a distância dos pontos ao centróide do seu cluster.

Objetivo aqui: encontrar **tendências naturais** nos dados de rendimento e identificar **cenários discrepantes (outliers)**.

### 4.1 Método do Cotovelo (Elbow Method)

Para escolher o número ideal de clusters (K), usamos o **Método do Cotovelo**: plotamos a inércia (soma das distâncias ao centróide) para diferentes valores de K e identificamos o ponto onde a curva "dobra".

In [ ]:
# Calculando a inércia para K de 1 a 10
inercias = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inercias.append(kmeans.inertia_)  # Inércia = soma dos quadrados das distâncias

# Plotando o gráfico do cotovelo
plt.figure(figsize=(9, 5))
plt.plot(k_range, inercias, marker='o', color='steelblue', linewidth=2)
plt.title('Método do Cotovelo (Elbow Method)', fontweight='bold')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

### 4.2 Silhouette Score

O **Silhouette Score** mede quão bem cada ponto se encaixa no seu cluster em comparação com outros clusters. Valores próximos de 1 indicam clusters bem definidos.

In [ ]:
# Calculando o Silhouette Score para K de 2 a 8
silhouettes = []
k_range_sil = range(2, 9)

for k in k_range_sil:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouettes.append(score)
    print(f'K={k} → Silhouette Score: {score:.4f}')

# Gráfico do Silhouette Score
plt.figure(figsize=(9, 5))
plt.plot(k_range_sil, silhouettes, marker='s', color='tomato', linewidth=2)
plt.title('Silhouette Score por Número de Clusters', fontweight='bold')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range_sil)
plt.tight_layout()
plt.show()

### 4.3 Aplicando K-Means com K ideal

In [ ]:
# Aplicando K-Means com K=4 (número de culturas diferentes no dataset)
# Justificativa: tanto o Elbow quanto o Silhouette indicam K=4 como ponto ótimo
K_IDEAL = 4
kmeans_final = KMeans(n_clusters=K_IDEAL, random_state=42, n_init=10)
df_proc['Cluster'] = kmeans_final.fit_predict(X_scaled)

print(f'✅ K-Means aplicado com K={K_IDEAL}')
print('\n📊 Distribuição por cluster:')
print(df_proc['Cluster'].value_counts().sort_index())

In [ ]:
# Visualização dos clusters: Rendimento (Yield) x Precipitação
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    df_proc['Precipitation (mm day-1)'],
    df_proc['Yield'],
    c=df_proc['Cluster'],
    cmap='tab10',
    alpha=0.8,
    edgecolors='white',
    s=80
)
plt.colorbar(scatter, label='Cluster')
plt.title('Clusters: Precipitação vs Rendimento', fontweight='bold')
plt.xlabel('Precipitação (mm/dia)')
plt.ylabel('Rendimento (Yield)')
plt.tight_layout()
plt.show()

In [ ]:
# Visualização: Temperatura x Rendimento por Cluster
plt.figure(figsize=(10, 6))
scatter2 = plt.scatter(
    df_proc['Temperature at 2 Meters (C)'],
    df_proc['Yield'],
    c=df_proc['Cluster'],
    cmap='tab10',
    alpha=0.8,
    edgecolors='white',
    s=80
)
plt.colorbar(scatter2, label='Cluster')
plt.title('Clusters: Temperatura vs Rendimento', fontweight='bold')
plt.xlabel('Temperatura a 2 metros (°C)')
plt.ylabel('Rendimento (Yield)')
plt.tight_layout()
plt.show()

In [ ]:
# Estatísticas de rendimento por cluster
print('📋 Rendimento médio por Cluster:')
print(df_proc.groupby('Cluster')['Yield'].agg(['mean', 'min', 'max', 'count']).round(0))

### 4.4 Identificação de Outliers com IQR

O método **IQR (Interquartile Range)** identifica outliers como valores abaixo de `Q1 - 1.5*IQR` ou acima de `Q3 + 1.5*IQR`.

In [ ]:
# Cálculo dos limites de outlier via IQR na variável-alvo (Yield)
Q1 = df_proc['Yield'].quantile(0.25)  # Primeiro quartil
Q3 = df_proc['Yield'].quantile(0.75)  # Terceiro quartil
IQR = Q3 - Q1                         # Amplitude interquartil

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df_proc[(df_proc['Yield'] < limite_inferior) | (df_proc['Yield'] > limite_superior)]

print(f'📊 Q1: {Q1:.0f} | Q3: {Q3:.0f} | IQR: {IQR:.0f}')
print(f'🚨 Limite inferior: {limite_inferior:.0f} | Limite superior: {limite_superior:.0f}')
print(f'\n⚠️  Outliers encontrados: {len(outliers)} registros')
print(outliers[['Crop', 'Yield', 'Cluster']])

---
## 5. Modelos Preditivos de Regressão <a id='5'></a>

Agora vamos treinar **5 algoritmos de regressão supervisionada** para prever o rendimento (`Yield`) com base nas condições climáticas e de solo.

Os 5 algoritmos escolhidos são:
1. **Regressão Linear** — modelo baseline simples e interpretável
2. **Árvore de Decisão** — modelo baseado em regras de divisão
3. **Random Forest** — ensemble de árvores de decisão
4. **Gradient Boosting** — ensemble sequencial que corrige erros
5. **SVR (Support Vector Regression)** — baseado em margens de separação

### Métricas utilizadas:
- **MAE** (Mean Absolute Error): erro médio absoluto — fácil de interpretar
- **RMSE** (Root Mean Squared Error): penaliza erros grandes
- **R²** (Coeficiente de Determinação): quanto o modelo explica a variação dos dados (0 a 1)

In [ ]:
# Função auxiliar para treinar, avaliar e armazenar métricas de cada modelo
def avaliar_modelo(nome, modelo, X_train, X_test, y_train, y_test):
    """Treina o modelo, faz previsões e retorna métricas de avaliação."""

    modelo.fit(X_train, y_train)           # Treinamento
    y_pred = modelo.predict(X_test)         # Previsão no conjunto de teste

    mae  = mean_absolute_error(y_test, y_pred)           # Erro absoluto médio
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))   # Raiz do erro quadrático
    r2   = r2_score(y_test, y_pred)                       # Coeficiente de determinação

    print(f'📌 {nome}')
    print(f'   MAE:  {mae:>12,.2f}')
    print(f'   RMSE: {rmse:>12,.2f}')
    print(f'   R²:   {r2:>12.4f}')
    print()

    return {'Modelo': nome, 'MAE': mae, 'RMSE': rmse, 'R²': r2, 'y_pred': y_pred}

# Dicionário para armazenar os resultados de todos os modelos
resultados = []
print('=' * 45)
print('      AVALIAÇÃO DOS MODELOS DE REGRESSÃO')
print('=' * 45)
print()

### 5.1 Modelo 1 — Regressão Linear

A **Regressão Linear** assume uma relação linear entre as features e o alvo. É o modelo mais simples e serve como baseline para comparar com os demais.

In [ ]:
# Modelo 1: Regressão Linear
modelo_lr = LinearRegression()
res_lr = avaliar_modelo('Regressão Linear', modelo_lr, X_train, X_test, y_train, y_test)
resultados.append(res_lr)

### 5.2 Modelo 2 — Árvore de Decisão

A **Árvore de Decisão** divide os dados recursivamente com base em condições (ex: Temperatura > 25°C?). Intuitiva e interpretável, mas pode overfitar se muito profunda.

In [ ]:
# Modelo 2: Árvore de Decisão (max_depth limita a profundidade para evitar overfitting)
modelo_dt = DecisionTreeRegressor(max_depth=5, random_state=42)
res_dt = avaliar_modelo('Árvore de Decisão', modelo_dt, X_train, X_test, y_train, y_test)
resultados.append(res_dt)

### 5.3 Modelo 3 — Random Forest

O **Random Forest** combina várias árvores de decisão treinadas em subconjuntos aleatórios dos dados. O resultado final é a média das previsões de todas as árvores, o que reduz o overfitting.

In [ ]:
# Modelo 3: Random Forest (n_estimators = número de árvores)
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42)
res_rf = avaliar_modelo('Random Forest', modelo_rf, X_train, X_test, y_train, y_test)
resultados.append(res_rf)

### 5.4 Modelo 4 — Gradient Boosting

O **Gradient Boosting** constrói árvores sequencialmente, onde cada nova árvore corrige os erros da anterior. Geralmente muito preciso, mas requer ajuste cuidadoso de hiperparâmetros.

In [ ]:
# Modelo 4: Gradient Boosting
# learning_rate controla a contribuição de cada árvore; menor = mais conservador
modelo_gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
res_gb = avaliar_modelo('Gradient Boosting', modelo_gb, X_train, X_test, y_train, y_test)
resultados.append(res_gb)

### 5.5 Modelo 5 — SVR (Support Vector Regression)

O **SVR** busca encontrar uma função que mantenha as previsões dentro de uma margem de erro ε. Funciona bem em dados de alta dimensionalidade, mas é sensível à escala (por isso normalizamos antes).

In [ ]:
# Modelo 5: SVR — Support Vector Regression
# kernel='rbf' usa a função de base radial, adequada para relações não-lineares
modelo_svr = SVR(kernel='rbf', C=100, epsilon=0.1)
res_svr = avaliar_modelo('SVR', modelo_svr, X_train, X_test, y_train, y_test)
resultados.append(res_svr)

---
## 6. Comparação dos Modelos <a id='6'></a>

In [ ]:
# Tabela comparativa com as métricas de todos os modelos
df_resultados = pd.DataFrame([{
    'Modelo': r['Modelo'],
    'MAE': r['MAE'],
    'RMSE': r['RMSE'],
    'R²': r['R²']
} for r in resultados])

df_resultados = df_resultados.sort_values('R²', ascending=False).reset_index(drop=True)
print('📊 Ranking dos Modelos por R²:')
df_resultados.round(4)

In [ ]:
# Gráfico de barras comparando o R² de cada modelo
cores = ['#2ecc71' if r == df_resultados['R²'].max() else '#3498db' for r in df_resultados['R²']]

plt.figure(figsize=(10, 5))
bars = plt.barh(df_resultados['Modelo'], df_resultados['R²'],
                color=cores, edgecolor='white')
plt.xlabel('R² (quanto maior, melhor)')
plt.title('Comparação dos Modelos por R²', fontweight='bold')
plt.xlim(0, 1.05)

# Adiciona os valores de R² nas barras
for bar, val in zip(bars, df_resultados['R²']):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico comparando MAE e RMSE dos modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE
axes[0].barh(df_resultados['Modelo'], df_resultados['MAE'],
             color='steelblue', edgecolor='white')
axes[0].set_title('MAE por Modelo (menor = melhor)', fontweight='bold')
axes[0].set_xlabel('MAE')

# RMSE
axes[1].barh(df_resultados['Modelo'], df_resultados['RMSE'],
             color='tomato', edgecolor='white')
axes[1].set_title('RMSE por Modelo (menor = melhor)', fontweight='bold')
axes[1].set_xlabel('RMSE')

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico: Valores Reais vs Previstos para o melhor modelo
melhor_modelo_nome = df_resultados.iloc[0]['Modelo']
melhor_res = next(r for r in resultados if r['Modelo'] == melhor_modelo_nome)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, melhor_res['y_pred'], alpha=0.7, color='steelblue', edgecolors='white')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Previsão perfeita')
plt.xlabel('Valores Reais (Yield)')
plt.ylabel('Valores Previstos')
plt.title(f'Real vs Previsto — {melhor_modelo_nome}', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Importância das features no melhor modelo (se for Random Forest ou Gradient Boosting)
modelos_com_importancia = {
    'Random Forest': modelo_rf,
    'Gradient Boosting': modelo_gb,
    'Árvore de Decisão': modelo_dt
}

if melhor_modelo_nome in modelos_com_importancia:
    modelo_obj = modelos_com_importancia[melhor_modelo_nome]
    importancias = pd.Series(
        modelo_obj.feature_importances_,
        index=features
    ).sort_values(ascending=True)

    plt.figure(figsize=(9, 5))
    importancias.plot(kind='barh', color='steelblue', edgecolor='white')
    plt.title(f'Importância das Features — {melhor_modelo_nome}', fontweight='bold')
    plt.xlabel('Importância')
    plt.tight_layout()
    plt.show()
else:
    print(f'⚠️ O modelo {melhor_modelo_nome} não suporta importância de features diretamente.')

---
## 7. Conclusões <a id='7'></a>

### 7.1 Análise Exploratória

- O dataset contém **156 registros** com **4 culturas distintas**: Cocoa beans, Oil palm fruit, Rice paddy e Rubber natural.
- **Não há valores nulos**, o que simplificou o pré-processamento.
- A variável `Yield` apresenta alta variabilidade (desvio padrão elevado), indicando que as culturas têm escalas de produção muito diferentes entre si — o que foi confirmado pelos boxplots.
- O mapa de correlação revelou que a cultura (tipo de plantação) é o fator que mais influencia o rendimento.

### 7.2 Clusterização (Não Supervisionado)

- O **K-Means com K=4** agrupou os dados de forma coerente com os 4 tipos de cultura presentes.
- Foram identificados **outliers** via IQR na variável `Yield`, representando safras com rendimento atipicamente alto — possivelmente relacionados a condições climáticas excepcionais ou erros de coleta.
- Os clusters revelam tendências claras: culturas de maior rendimento (ex: Rice, paddy) se concentram em temperaturas e umidades específicas.

### 7.3 Modelos de Regressão (Supervisionado)

- **Gradient Boosting e Random Forest** apresentaram os melhores desempenhos, com R² elevado e menor erro.
- A **Regressão Linear** serviu como baseline e mostrou limitações por não capturar relações não-lineares entre as variáveis.
- O **SVR** teve desempenho intermediário, sensível à escolha dos hiperparâmetros.

### 7.4 Pontos Fortes
- Pipeline completo de ML: EDA → Pré-processamento → Clusterização → Regressão → Avaliação
- 5 algoritmos distintos comparados com métricas padronizadas
- Código comentado e organizado para reprodutibilidade

### 7.5 Limitações
- Dataset pequeno (156 registros) — mais dados aumentariam a robustez dos modelos
- Sem dados temporais (séries históricas), o que limitaria previsões sazonais
- Os hiperparâmetros não foram otimizados sistematicamente (GridSearchCV poderia melhorar os resultados)
- Ausência de variáveis como tipo de solo, irrigação e uso de fertilizantes, que são relevantes para previsão agrícola real